In [2]:
import pandas as pd
import os
import zipfile
import json
import random
from PIL import Image

from tqdm import tqdm
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision import transforms
from itertools import combinations, product
import torchvision
import matplotlib.pyplot as plt
import numpy as np
from torch.autograd import Variable
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import torchvision.models as models

In [3]:
# from numba import cuda
# cuda.select_device(0)
# cuda.close()
# cuda.select_device(0)

In [4]:
with open('/kaggle/input/deepfake/train/meta.json', 'r', encoding='utf-8') as f:
    meta = json.load(f)  # meta теперь обычный словарь (dict)

In [5]:
id_to_images = {}  # ключ: id (строка '000000'), значение: список [(filename, label), ...]
root_dir = '/kaggle/input/deepfake/train/images'


for key, value in meta.items():
    person_id = key.split('/')[0]   # "000000"
    filename  = key.split('/')[1]  # "0.jpg"
    if person_id not in id_to_images:
        id_to_images[person_id] = []
    id_to_images[person_id].append((filename, value))

all_ids = list(id_to_images.keys())

def create_triplets(id_to_images):
    triplets = []
    all_ids = list(id_to_images.keys())

    for person_id in all_ids:
        # Получаем все изображения для текущего человека
        imgs = id_to_images[person_id]

        # Выбираем реальные изображения (label == 0)
        real_imgs = [img for img in imgs if img[1] == 0]

        # Если реальных изображений меньше 2, пропускаем
        if len(real_imgs) < 2:
            continue

        # Создаем пары (anchor, positive)
        for anchor, positive in combinations(real_imgs, 2):
            # Выбираем negative из другого класса
            negative_id = random.choice([i for i in all_ids if i != person_id])
            negative_imgs = id_to_images[negative_id]
            negative = random.choice([img for img in negative_imgs if img[1] == 0])

            # Добавляем triplet с полным путем
            anchor_path = os.path.join(person_id, anchor[0])  # Пример: '00000000/0.jpg'
            positive_path = os.path.join(person_id, positive[0])  # Пример: '00000000/1.jpg'
            negative_path = os.path.join(negative_id, negative[0])  # Пример: '00000001/0.jpg'

            triplets.append((anchor_path, positive_path, negative_path))

    return triplets

# Создание триплетов
triplets = create_triplets(id_to_images)
print(f"Создано {len(triplets)} триплетов")


Создано 91552 триплетов


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

'''Модель CNN'''
# class FaceCNN(nn.Module):
#     def __init__(self, embedding_size=128):
#         super(FaceCNN, self).__init__()
#         self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
#         self.bn1 = nn.BatchNorm2d(32)
#         self.dropout1 = nn.Dropout(0.5)  # Dropout с вероятностью 0.5
#         self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
#         self.bn2 = nn.BatchNorm2d(64)
#         self.dropout2 = nn.Dropout(0.5)  # Dropout с вероятностью 0.5
#         self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
#         self.fc = nn.Linear(64, embedding_size)

#     def forward(self, x):
#         x = self.dropout1(F.relu(self.bn1(self.conv1(x))))
#         x = self.dropout2(F.relu(self.bn2(self.conv2(x))))
#         x = self.global_avg_pool(x)
#         x = x.view(x.size(0), -1)  # Flatten
#         x = self.fc(x)
#         return x

'Модель CNN'

In [7]:
'''resnet18'''

from torchvision import models

class FaceCNN(nn.Module):
    def __init__(self, embedding_size=128):
        super(FaceCNN, self).__init__()
        self.backbone = models.resnet18(pretrained=True)  # Используем предобученную ResNet18
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, embedding_size)

    def forward(self, x):
        return self.backbone(x)

In [8]:
class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(TripletLoss, self).__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        distance_positive = F.pairwise_distance(anchor, positive)
        distance_negative = F.pairwise_distance(anchor, negative)
        losses = F.relu(distance_positive - distance_negative + self.margin)
        return losses.mean()

In [9]:
# torch.cuda.empty_cache()

In [10]:
from torch.utils.data import DataLoader
from torch.optim import Adam
from tqdm import tqdm

# Инициализация модели и оптимизатора

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = 'cpu'
model = FaceCNN().to(device)
optimizer = Adam(model.parameters(), lr=1e-2)
criterion = TripletLoss(margin=1.0)


class TripletFaceDataset(Dataset):
    def __init__(self, triplets, root_dir, transform=None):
        self.triplets = triplets
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        anchor_path, positive_path, negative_path = self.triplets[idx]

        # Загрузка изображений с учетом вложенных папок
        anchor_img = Image.open(os.path.join(self.root_dir, anchor_path)).convert('RGB')
        positive_img = Image.open(os.path.join(self.root_dir, positive_path)).convert('RGB')
        negative_img = Image.open(os.path.join(self.root_dir, negative_path)).convert('RGB')

        # Применение трансформаций
        if self.transform:
            anchor_img = self.transform(anchor_img)
            positive_img = self.transform(positive_img)
            negative_img = self.transform(negative_img)

        return anchor_img, positive_img, negative_img


# Трансформации
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Создание Dataset
root_dir = '/kaggle/input/deepfake/train/images'
dataset = TripletFaceDataset(triplets, root_dir, transform=transform)

# Создание DataLoader
loader = DataLoader(dataset, batch_size=128, shuffle=True)


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 196MB/s]


In [11]:
epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = 'cpu'


for epoch in tqdm(range(epochs)):
    model.train()
    for anchor, positive, negative in loader:
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

        # Получение эмбеддингов
        anchor_emb = model(anchor)
        positive_emb = model(positive)
        negative_emb = model(negative)

        # Вычисление потерь
        loss = criterion(anchor_emb, positive_emb, negative_emb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")

 20%|██        | 1/5 [22:55<1:31:41, 1375.41s/it]

Epoch 1/5, Loss: 0.7795896530151367


 40%|████      | 2/5 [38:02<54:58, 1099.66s/it]  

Epoch 2/5, Loss: 0.29082387685775757


 60%|██████    | 3/5 [53:04<33:38, 1009.41s/it]

Epoch 3/5, Loss: 0.48457151651382446


 80%|████████  | 4/5 [1:08:04<16:06, 966.38s/it]

Epoch 4/5, Loss: 0.20911721885204315


100%|██████████| 5/5 [1:22:54<00:00, 994.85s/it]

Epoch 5/5, Loss: 0.22981032729148865


In [13]:
import joblib
joblib.dump(model, 'CNN_2.joblib')

['CNN_2.joblib']

In [12]:
torch.save(model.state_dict(), 'CNN_2.pth')

## Testing

In [17]:
import torch
import torch.nn as nn
from torchvision import models

# Определение модели
class FaceCNN(nn.Module):
    def __init__(self, embedding_size=128):
        super(FaceCNN, self).__init__()
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, embedding_size)

    def forward(self, x):
        return self.backbone(x)

# Загрузка модели
model = FaceCNN(embedding_size=128)
model.load_state_dict(torch.load("CNN_2.pth"))
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()  # Переводим модель в режим оценки

<ipython-input-17-1359f2cdc713>:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("CNN_2.pth"))


FaceCNN(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_ru

In [ ]:
from torchvision import transforms
from PIL import Image

# Трансформации (должны быть такими же, как при обучении)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Изменение размера до 224x224
    transforms.ToTensor(),  # Преобразование в тензор
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Нормализация
])

# Загрузка нового изображения
def load_image(image_path):
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0)  # Добавляем batch dimension
    return img_tensor.to('cuda' if torch.cuda.is_available() else 'cpu')

# Пример загрузки нового изображения
new_image_tensor = load_image("path_to_new_image.jpg")

In [ ]:
# model.load_state_dict(torch.load("/kaggle/input/cnn_deepfake/tensorflow2/cnn/1/CNN_1.pth", map_location='cpu'))

In [18]:
import os

test_dir = "/kaggle/input/test-deepfake/test_public/images"
test_pairs = []

# Сбор пар изображений
for pair_id in os.listdir(test_dir):
    img1_path = os.path.join(test_dir, pair_id, "0.jpg")
    img2_path = os.path.join(test_dir, pair_id, "1.jpg")
    test_pairs.append((pair_id, img1_path, img2_path))

print(f"Найдено {len(test_pairs)} пар для тестирования")

Найдено 161191 пар для тестирования


In [20]:
import torch
import torch.nn.functional as F

def extract_embedding(model, image_path, transform, device):
    """Извлекает эмбеддинг для одного изображения."""
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)  # Добавляем batch dimension
    with torch.no_grad():
        embedding = model(img_tensor)
    return embedding


In [21]:
def cosine_similarity(embedding1, embedding2):
    """Вычисляет косинусную схожесть между двумя эмбеддингами."""
    return F.cosine_similarity(embedding1, embedding2).item()

# similarity = cosine_similarity(embedding1, embedding2)
# print(f"Косинусная схожесть: {similarity}")

По выгрузке ниже, делаем вывод о распределении предсказанных значений. Они все очень близки к 1. Модель resnet18,также как и CNN не очень хорошо работает, так как слишком часто склоняется к 1 

In [ ]:
results = []

for pair_id, img1_path, img2_path in test_pairs:
    # Извлечение эмбеддингов
    embedding1 = extract_embedding(model, img1_path, transform, device)
    embedding2 = extract_embedding(model, img2_path, transform, device)

    # Вычисление косинусной схожести
    similarity = cosine_similarity(embedding1, embedding2)

    # Сохранение результата
    results.append((pair_id, similarity))
    print(f"Pair ID: {pair_id}, Cosine Similarity: {similarity}")

# # Вывод результатов
# for pair_id, similarity in results:
#     print(f"Pair ID: {pair_id}, Cosine Similarity: {similarity}")

Pair ID: 00071595, Cosine Similarity: 0.9976626038551331
Pair ID: 00017832, Cosine Similarity: 0.9993042945861816
Pair ID: 00093173, Cosine Similarity: 0.9934124946594238
Pair ID: 00075969, Cosine Similarity: 0.9947448372840881
Pair ID: 00135956, Cosine Similarity: 0.9982755184173584
Pair ID: 00097186, Cosine Similarity: 0.9987053275108337
Pair ID: 00114148, Cosine Similarity: 0.9985228180885315
Pair ID: 00160393, Cosine Similarity: 0.9978188276290894
Pair ID: 00109845, Cosine Similarity: 0.9959602355957031
Pair ID: 00026398, Cosine Similarity: 0.9974139928817749
Pair ID: 00104106, Cosine Similarity: 0.9981894493103027
Pair ID: 00141665, Cosine Similarity: 1.0000001192092896
Pair ID: 00161059, Cosine Similarity: 0.9994584321975708
Pair ID: 00031353, Cosine Similarity: 0.9999999403953552
Pair ID: 00075776, Cosine Similarity: 0.9934401512145996
Pair ID: 00025351, Cosine Similarity: 0.9977080821990967
Pair ID: 00120866, Cosine Similarity: 0.9986931085586548
Pair ID: 00040647, Cosine Simil

In [ ]:
import pandas as pd

df_results = pd.DataFrame(results, columns=["pair_id", "cosine_similarity"])
df_results.to_csv("test_results.csv", index=False)